![earthkit-data-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-data-light.svg)

# Earthkit Data - GRIB

In this notebook you will see how to:

- inspect GRIB data
- work with fields and fieldlists
- modify fields
- convert GRIB to Xarray
- convert GRIB to Pandas

## Getting the data

First we read some GRIB data from disk with [from_source](https://earthkit-data/readthedocs.io/en/latest/concepts/inputs/from_source.html). The returned object provides some basic information but its primary goal is to convert the data into the required representation for further work. The actual data loading is deferred as much as possible, until the data is converted into a given type.

In [1]:
import earthkit.data as ekd

d = ekd.from_source("file", "test4.grib")
d

path,../test4.grib
size,509.5 KiB
types,"fieldlist, pandas, xarray, numpy, array"


In [2]:
d.available_types

['fieldlist', 'pandas', 'xarray', 'numpy', 'array']

## Fieldlists and fields

GRIB data can be converted into a [FieldList](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList), which represents each GRIB message as a field. In earthkit a field is a horizontal slice of the atmosphere at a given time. In this sense the Field object is generic enough to be able to represent other types of data than GRIB (e.g. NetCDF, GeoTIFF, dictionary data etc.)

In [3]:
# fl is a fieldlist
fl = d.to_fieldlist()

In [4]:
# fl contains 4 fields
len(fl)

4

[ls()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList.ls) lists the fields in the fieldlist.

In [5]:
fl.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
2,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll
3,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


In [6]:
# we can iterate through the fields
for f in fl: 
    print(f)

Field(t, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 500, pressure, 0, regular_ll)
Field(z, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 500, pressure, 0, regular_ll)
Field(t, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)
Field(z, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 850, pressure, 0, regular_ll)


In [7]:
# f is the first field in the fieldlist
f = fl[0]
print(f)

Field(t, 2007-01-01 12:00:00, 2007-01-01 12:00:00, 0:00:00, 500, pressure, 0, regular_ll)


#### Fields metadata

A Field is composed of format independent components, each containing a set of metadata keys. The most important components and keys are as follows:

- parameter:
    - parameter.variable
    - parameter.units
    - parameter.chem_variable
- time:
    - time.base_datetime
    - time.valid_datetime
    - time.step
- vertical:
    - vertical.level
    - vertical.level_type
- geography:
    - geography.latitudes
    - geography.longitudes
    - geography.shape
- ensemble:
    - ensemble.member

The metadata can be accessed via the [get()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.get) method.

In [8]:
f.get("time.base_datetime")

datetime.datetime(2007, 1, 1, 12, 0)

We can also use the key methods on the field components.

In [9]:
f.time.base_datetime()

datetime.datetime(2007, 1, 1, 12, 0)

The components contain format independent metadata. This is most obvious when looking at the level type.

In [10]:
# the GRIB metadata would be isobaricInhPa
f.vertical.level_type()

'pressure'

Since the field was created from GRIB the raw (ecCodes) GRIB metadata is also available via the [metadata()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.metadatametadata) method.

In [11]:
f.metadata("typeOfLevel")

'isobaricInhPa'

The [get()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.get) method also works for raw metadata but the keys must be prefixed with ``"metadata."``.

In [12]:
f.get("metadata.typeOfLevel")

'isobaricInhPa'

#### Field units

Units are represented by an object that provides normalisation and can used in comparisons (to a string or another Units object).

In [13]:
u = f.parameter.units()
u

<Unit('kelvin')>

In [14]:
u == "kelvin", u == "K"

(True, True)

When possible the Units object is based on a ``pint.Units`` object that can be directly accessed when needed.

In [15]:
u_p = u.to_pint()
type(u_p)

pint.Unit

#### Field overview

To get an overview about the components/keys of a field simply use the automatic display.

In [16]:
# this is the same as f.describe()
f

number_of_values,65160
array_type,ndarray
array_dtype,float64
variable,t
standard_name,air_temperature
long_name,Temperature
units,kelvin
chem_variable,None
valid_datetime,2007-01-01 12:00:00
base_datetime,2007-01-01 12:00:00
step,0:00:00


#### Field values

Use [to_numpy](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.to_numpy) to access the field values as a numpy array. By default it uses the shape of the field.

In [17]:
a = f.to_numpy()
a.shape, a

((181, 360),
 array([[228.04600525, 228.04600525, 228.04600525, ..., 228.04600525,
         228.04600525, 228.04600525],
        [228.60850525, 228.57920837, 228.5499115 , ..., 228.698349  ,
         228.66807556, 228.63877869],
        [228.33311462, 228.26768494, 228.20420837, ..., 228.53623962,
         228.46788025, 228.39952087],
        ...,
        [244.47471619, 244.41807556, 244.36045837, ..., 244.64268494,
         244.58702087, 244.53135681],
        [245.13487244, 245.1124115 , 245.088974  , ..., 245.20323181,
         245.18077087, 245.15830994],
        [246.16319275, 246.16319275, 246.16319275, ..., 246.16319275,
         246.16319275, 246.16319275]], shape=(181, 360)))

The [values](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.values) property is also available. Please note this is always return a flattened array. 

#### Field geography

In [18]:
# the shape is based on the geography (when available)
f.shape, f.geography.shape()

((181, 360), (181, 360))

We can use [latlons()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/field/component/geography/index.html#earthkit.data.field.component.geography.GeographyBase.latlons) to get a tuple of the latitudes and longitudes arrays. Please note "geography.latlons" cannot be used as a key.

In [19]:
f.geography.latlons()

(array([[ 90.,  90.,  90., ...,  90.,  90.,  90.],
        [ 89.,  89.,  89., ...,  89.,  89.,  89.],
        [ 88.,  88.,  88., ...,  88.,  88.,  88.],
        ...,
        [-88., -88., -88., ..., -88., -88., -88.],
        [-89., -89., -89., ..., -89., -89., -89.],
        [-90., -90., -90., ..., -90., -90., -90.]], shape=(181, 360)),
 array([[  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.],
        ...,
        [  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.],
        [  0.,   1.,   2., ..., 357., 358., 359.]], shape=(181, 360)))

The new way of describing grids is to use a grid spec. It is available as a dict from the field.

In [20]:
f.geography.grid_spec()

{'grid': [1, 1]}

## Modifying fields

Fields can be modified by [set()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.set), which generates a new field with updated values/components.

In [21]:
vals = f.values + 2.0
f1 = f.set({"vertical.level": 700, "values": vals})
f1.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,700,pressure,0,regular_ll


In [22]:
# compare the metadata in the old and new fields
f.get("vertical.level"), f1.get("vertical.level")

(500, 700)

In [23]:
# compare the values in the old and new fields
f.values.max(), f1.values.max()

(np.float64(273.33799743652344), np.float64(275.33799743652344))

See also the following more elaborate examples for details:

- [modifying metadata](https://earthkit-data.readthedocs.io/en/latest/how-tos/grib/grib_modify_metadata.html)
- [modifying values](https://earthkit-data.readthedocs.io/en/latest/how-tos/grib/grib_modify_values.html)

## Fieldlist subsetting

Use [sel()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList.sel)
to select the fields matching the given metadata conditions.

In [24]:
fl.sel({"parameter.variable": "t"}).ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,t,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


## Arithmetics

In [25]:
# select geopotential fields
z = fl.sel({"parameter.variable": "z"})

We compute the geopotential height using fieldlist arithmetic. The result will be a new fieldlist entirely stored in memory.

In [26]:
# compute the geopotential height
# h and z are fieldlists
h = z / 9.81

# check the results
for zv, hv in zip(z, h):
    print(f"z-max: {zv.values.max()} h-max: {hv.values.max()}")

z-max: 58345.609375 h-max: 5947.564666156983
z-max: 16578.98828125 h-max: 1690.0089991080529


The metadata in the new "h" fields are still the same as in the "z" fields. We can correct it manually.

In [27]:
h.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,z,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


In [28]:
h = h.set({"parameter.variable": "gh", "parameter.units": "gpm"})
h.ls()

,parameter.variable,time.valid_datetime,time.base_datetime,time.step,vertical.level,vertical.level_type,ensemble.member,geography.grid_type
0,gh,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,500,pressure,0,regular_ll
1,gh,2007-01-01 12:00:00,2007-01-01 12:00:00,0 days,850,pressure,0,regular_ll


Fields can also be used in arithmetic expressions.

In [29]:
# compute the geopotential height
# h and z_first are fields
z_first = z[0]
h = z[0] / 9.81

f"z-max: {z_first.values.max()} h-max: {h.values.max()}"

'z-max: 58345.609375 h-max: 5947.564666156983'

## Converting to Xarray

For this exercise we fetch another GRIB file containing more variation in time and level.

In [30]:
# get GRIB data containing variations in date/time/step and level
d = ekd.from_source("file", "../pl.grib")
fl = d.to_fieldlist()
fl.unique("parameter.variable", "time.base_datetime", "time.step", "vertical.level")

{'parameter.variable': ('t', 'r'),
 'time.base_datetime': (datetime.datetime(2024, 6, 3, 0, 0),
  datetime.datetime(2024, 6, 3, 12, 0),
  datetime.datetime(2024, 6, 4, 0, 0),
  datetime.datetime(2024, 6, 4, 12, 0)),
 'time.step': (datetime.timedelta(0), datetime.timedelta(seconds=21600)),
 'vertical.level': (700, 500)}

We can convert fieldlists to Xarray using earthkit-data's own Xarray engine. It comes with a very large number of options, but for our data the defaults are perfectly fine.

In [31]:
ds = fl.to_xarray()
ds

<xarray.Dataset> Size: 176kB
Dimensions:                  (forecast_reference_time: 4, step: 2, level: 2,
                              latitude: 19, longitude: 36)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 32B 202...
  * step                     (step) timedelta64[ns] 16B 00:00:00 06:00:00
  * level                    (level) int64 16B 500 700
  * latitude                 (latitude) float64 152B 90.0 80.0 ... -80.0 -90.0
  * longitude                (longitude) float64 288B 0.0 10.0 ... 340.0 350.0
Data variables:
    r                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
    t                        (forecast_reference_time, step, level, latitude, longitude) float64 88kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

The Xarray we created uses the "forecasts" time dimension mode, with ``forecast_reference_time`` and ``step`` chosen as the temporal dimensions. The input data also forms a hypercube if we choose the valid datetime as the temporal dimension, so we can use the ``time_mode_dim`` option as follows:

In [32]:
ds = fl.to_xarray(time_dims="valid_time")
ds

<xarray.Dataset> Size: 176kB
Dimensions:     (valid_time: 8, level: 2, latitude: 19, longitude: 36)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 64B 2024-06-03 ... 2024-06-04T18:...
  * level       (level) int64 16B 500 700
  * latitude    (latitude) float64 152B 90.0 80.0 70.0 ... -70.0 -80.0 -90.0
  * longitude   (longitude) float64 288B 0.0 10.0 20.0 ... 330.0 340.0 350.0
Data variables:
    r           (valid_time, level, latitude, longitude) float64 88kB ...
    t           (valid_time, level, latitude, longitude) float64 88kB ...
Attributes:
    Conventions:  CF-1.8
    institution:  ECMWF

Xarrays created with earthkit-data have the ``earthkit`` accessor. It is an experimental feature and for each DataArray it stores earthkit specific metadata.

In [33]:
ds["t"].earthkit.grid_spec

{'grid': [10, 10]}

## Converting to Pandas

In [34]:
fl[0].to_pandas()

,lat,lon,value,datetime
0,90.0,0.0,265.464554,2024-06-03
1,90.0,10.0,265.464554,2024-06-03
2,90.0,20.0,265.464554,2024-06-03
3,90.0,30.0,265.464554,2024-06-03
4,90.0,40.0,265.464554,2024-06-03
...,...,...,...,...
679,-90.0,310.0,237.597366,2024-06-03
680,-90.0,320.0,237.597366,2024-06-03
681,-90.0,330.0,237.597366,2024-06-03
682,-90.0,340.0,237.597366,2024-06-03
